# PA Appeals & Policy Search — Path B Migration Plan

> **Status:** Planning — not yet started | **Decision date:** 2026-06-25

---

## 1. Executive Summary

Migrate the deployed `pa-appeals-and-policy-search` Databricks App from its current Python/Gradio stack to the Node.js/TypeScript/React stack in `Github-pa-appeals-and-policy-search/`. Preserve all existing functionality (semantic vector search, admin panel, PDF pipeline) while making WestLaw-style deterministic search the primary, front-and-center interface. The pipeline notebook is never touched.

## 2. Source Inventory

### 2a. Live Deployed App

| Item | Value |
| --- | --- |
| Source folder | `/Workspace/Users/0492734585@fema.dhs.gov/pa-appeals-and-policy-search/` |
| Runtime | Python 3 + Gradio |
| Entry point | `app.py` (244 lines) |
| App manifest | `app.yaml` — `command: [python, app.py]` |
| App status | RUNNING, deployment SUCCEEDED 2026-06-25T22:18:27Z |
| App URL | `https://pa-appeals-and-policy-search-5672234203219303.3.azure.databricksapps.com` |

**Features in the live app:**

- **Semantic search tab** — POSTs to VS REST API with columns `[chunk_id, filename, page_number, chunk_type, chunk_text]`. Passes the user's `X-Forwarded-Access-Token` directly (preserves per-user data access control). Returns relevance score, document name, page number, 800-char chunk text.
- **Admin tab** — three sub-functions:
  - *Index stats*: GET VS index → indexed row count, ready flag, status message
  - *Trigger refresh*: POST `jobs/runs/submit` — one-time run of the processor notebook with `pypdf` dependency
  - *Check run status*: GET `jobs/runs/get` → lifecycle state, result state, start/end timestamps
- **Pipeline notebook** (called by admin refresh — **never moved or edited**):
  - Path: `/Users/0492734585@fema.dhs.gov/pa-appeals-and-policy-search/PA Appeals PDF Incremental Processor`
  - Recursively scans `/Volumes/tws_ro_region5/rcd/pa_second_appeals` for new/modified PDFs
  - Parses with `pypdf`, appends page-level chunks to `tws_ro_region5.rcd.pa_appeals_chunks_vs`
  - Triggers VS sync via `WorkspaceClient().vector_search_indexes.sync_index()`
  - As of last run: 2,871 total PDFs, 2,797 processed, 63 newly appended
  - Final cell: **Power Automate setup guide** (markdown only, never executed — documents the SharePoint-triggered flow that auto-submits this notebook when new PDFs land)

**Confirmed data assets:**

| Asset | Value |
| --- | --- |
| Vector Search Index | `tws_ro_region5.rcd.pa_appeals_chunks_vs_index` |
| VS Endpoint | `pa-appeals-search-endpoint` |
| Chunks table | `tws_ro_region5.rcd.pa_appeals_chunks_vs` |
| Volume (PDFs) | `/Volumes/tws_ro_region5/rcd/pa_second_appeals` |
| Embeddings | GTE-Large (managed by VS endpoint) |
| DATABRICKS_HOST | `https://adb-5672234203219303.3.azuredatabricks.net` |

---

### 2b. GitHub Repo (Code Reference Only)

> The git-tracked folder at `Github-pa-appeals-and-policy-search/` is used **as a code reference only**. No files are copied from it. All new files are authored directly in `pa-appeals-and-policy-search/` to avoid git-tracked folder permissions issues.

| Item | Value |
| --- | --- |
| Reference folder | `/Workspace/Users/0492734585@fema.dhs.gov/Github-pa-appeals-and-policy-search/` |
| Runtime | Node.js 18+ / TypeScript / React 18 / Vite / Express |
| Entry point | `app.yaml` — `command: [npm, run, start]`; `prestart` builds `tsc` + Vite |
| Build status | `npm run build` and `npm run typecheck` both pass |
| Tests | 22 query parser/evaluator + 10 path-guard — all green |
| Current mode | Demo (fabricated in-memory, no real data configured) |

**Repo features:**
- Deterministic search engine: lexer + recursive-descent parser, exact phrase / AND / OR / NOT / `NEAR(n)` / `ONEAR(n)`
- Three-panel React UI: SearchPanel | ResultsList | PdfReader (PDF.js, open-to-page, Prev/Next Match)
- Demo mode: fully functional offline with in-process PDF generation
- Pilot/Production SQL mode: parameterized queries against two Delta index tables via SQL warehouse
- Security: CSP `default-src 'self'`; `/pdf/:id` path-traversal guard (browser never supplies a file path)

## 3. Target Architecture

```
Browser (React + PDF.js)
├─ Tab: Search  <── DEFAULT, front and center
│    ├─ Mode toggle: Deterministic (WestLaw-style) | Semantic (AI similarity)
│    ├─ Left panel:   SearchPanel  — query input, mode toggle, syntax drawer, query history
│    ├─ Middle panel: ResultsList  — ranked hits, document grouping, match type badges
│    └─ Right panel:  PdfReader    — open-to-page, highlight, Prev/Next match
└─ Tab: Admin
     ├─ Index stats  (VS index row count, ready status)
     ├─ Trigger refresh  (one-time job run of processor notebook)
     └─ Run status check (polls while RUNNING)

Express server (Node.js / TypeScript)
├─ /api/search               → deterministic engine        (existing)
├─ /api/semantic-search      → proxies VS REST API          (new)
├─ /api/admin/stats          → proxies VS index GET         (new)
├─ /api/admin/refresh        → proxies jobs/runs/submit     (new)
├─ /api/admin/status/:runId  → proxies jobs/runs/get        (new)
├─ /api/status               → mode/index health            (existing)
└─ /pdf/:id                  → PDF stream                  (existing)

Unity Catalog
├─ tws_ro_region5.rcd.pa_appeals_chunks_vs_index    ← vector search (existing, live)
├─ tws_ro_region5.rcd.pa_appeals_chunks_vs           ← chunks table  (existing, live)
├─ [catalog].[schema].appeal_research_documents      ← deterministic index (to be built)
└─ [catalog].[schema].appeal_research_pages           ← deterministic index (to be built)

Volume (read-only)
└─ /Volumes/tws_ro_region5/rcd/pa_second_appeals     ← 2,871 PDFs (confirmed from notebook)

Notebook (NEVER MOVED OR EDITED)
└─ PA Appeals PDF Incremental Processor
    at: /Users/0492734585@fema.dhs.gov/pa-appeals-and-policy-search/
```

## 4. Target File Structure

All files are authored directly in `pa-appeals-and-policy-search/` (the live source folder). The git-tracked `Github-pa-appeals-and-policy-search/` folder is referenced for code logic only — nothing is copied from it. The Python files `app.py` and `requirements.txt` are replaced. The processor notebook stays at its existing path and is never edited.

```
pa-appeals-and-policy-search/          ← LIVE SOURCE FOLDER (all work happens here)
├── app.yaml                            ← REPLACED: Node.js command + all env vars
├── package.json                        ← AUTHORED in workspace
├── package-lock.json                   ← generated by npm install
├── vite.config.ts                      ← AUTHORED in workspace
├── MIGRATION_PLAN (this notebook)      ← planning doc
├── PA Appeals PDF Incremental Processor  ← UNTOUCHED notebook (stays here)
├── server/
│   ├── tsconfig.json
│   └── src/
│       ├── index.ts        ← AUTHORED: Express routes incl. /api/semantic-search, /api/admin/*
│       ├── config.ts       ← AUTHORED: env vars incl. VS_INDEX_NAME, PROCESSOR_NOTEBOOK_PATH
│       ├── data/           ← AUTHORED (logic matches repo reference)
│       ├── pdf/            ← AUTHORED (logic matches repo reference)
│       ├── search/         ← AUTHORED + wildcard/ATLEAST extensions
│       │   ├── queryParser.ts      ← + WILDCARD token, ATLEAST(n) token
│       │   ├── queryParser.test.ts ← + new test cases
│       │   ├── evaluator.ts        ← + wildcardSpans(), ATLEAST eval case
│       │   ├── normalize.ts
│       │   ├── searchService.ts
│       │   └── types.ts            ← + Wildcard/AtLeast QueryNode, "semantic" MatchType
│       └── semantic/       ← NEW
│           └── vectorSearch.ts  ← VS REST proxy, normalizes to SearchResult[]
├── client/
│   ├── index.html
│   ├── tsconfig.json
│   └── src/
│       ├── main.tsx
│       ├── App.tsx          ← tab layout (Search | Admin), Search default
│       ├── api.ts           ← + semanticSearch(), admin API calls
│       ├── types.ts         ← + AdminStats, AdminRunStatus, "semantic" MatchType
│       ├── styles.css       ← + tab, admin panel, mode toggle, grouping styles
│       ├── pdf/pdfSetup.ts
│       └── components/
│           ├── SearchPanel.tsx    ← mode toggle + query history (localStorage)
│           ├── ResultsList.tsx    ← document-level grouping toggle
│           ├── PdfReader.tsx
│           ├── AdvancedHelp.tsx   ← + wildcard and ATLEAST examples
│           ├── PilotBoundaries.tsx
│           ├── StatusBar.tsx
│           └── AdminPanel.tsx     ← NEW: ported from Gradio admin tab
└── notebooks/
    └── 01_build_page_index.py  ← AUTHORED (logic matches repo reference)
```

## 5. Build Phases

### Phase 1 — Scaffold

All files are created directly in `pa-appeals-and-policy-search/`. The `Github-pa-appeals-and-policy-search/` folder is kept open in a separate browser tab as a **read-only code reference** — no files are copied from it.

1. Remove `app.py` and `requirements.txt` from `pa-appeals-and-policy-search/` (no longer used).
2. Author the following directly in the workspace (using the Github repo as a reference for structure and logic):
   - `package.json` — same dependencies as the repo
   - `vite.config.ts` — same config as the repo
   - `server/tsconfig.json`, `client/tsconfig.json`
   - `server/src/` and `client/src/` trees (full scaffold, demo mode only at this stage)
3. Run `npm install` in the source folder to generate `package-lock.json`.
4. Confirm `npm run build` and `npm run typecheck` pass.

**Gate:** App builds cleanly in demo mode entirely from the workspace folder.

---

### Phase 2 — Backend: Semantic search proxy

Create `server/src/semantic/vectorSearch.ts`:
- Accepts `{ query, numResults, userToken }`
- POSTs to `${DATABRICKS_HOST}/api/2.0/vector-search/indexes/${VS_INDEX_NAME}/query`
- Authorization: `Bearer ${userToken}` forwarded from `req.headers['x-forwarded-access-token']` — **NOT the service principal token**
- Columns: `chunk_id`, `filename`, `page_number`, `chunk_type`, `chunk_text`
- Normalizes each row to `SearchResult` with `matchType: "semantic"`, raw float score, 500-char snippet, empty `highlightTerms`
- `matchExplanation`: `"Semantic match — relevance score: ${score.toFixed(3)}"`

Add `POST /api/semantic-search` to `index.ts`.

**Gate:** Route returns real results against the existing VS index.

---

### Phase 3 — Backend: Admin routes

All three routes forward `X-Forwarded-Access-Token` from the browser. No service-principal elevation.

| Route | Maps to |
| --- | --- |
| `GET /api/admin/stats` | `GET /api/2.0/vector-search/indexes/{VS_INDEX_NAME}` |
| `POST /api/admin/refresh` | `POST /api/2.1/jobs/runs/submit` with processor notebook task + pypdf dependency |
| `GET /api/admin/status/:runId` | `GET /api/2.1/jobs/runs/get?run_id={runId}` |

The refresh payload must exactly match the Python app (pypdf in env spec, `WORKSPACE` source, notebook path from config). Add `PROCESSOR_NOTEBOOK_PATH` to `config.ts`:
```
/Users/0492734585@fema.dhs.gov/pa-appeals-and-policy-search/PA Appeals PDF Incremental Processor
```

**Gate:** Admin routes return live data; a test refresh run completes successfully.

---

### Phase 4 — Query engine: WestLaw enhancements

#### Wildcard / Truncation (`administrat*`)
- **Lexer** (`queryParser.ts`): bare words ending in `*` → emit `WILDCARD` token with normalized prefix (strip `*`)
- **AST** (`types.ts`): add `{ type: "wildcard"; prefix: string }` to `QueryNode` union
- **Evaluator** (`evaluator.ts`): `wildcardSpans(prefix, tokens)` — match any token where `token.text.startsWith(prefix)`; contributes to `termHits`; all matching full-token values added to `literals` for highlight
- **AdvancedHelp**: add `{ syntax: 'administrat*', desc: 'Any term starting with "administrat" (truncation).' }`

#### ATLEAST(n) Frequency Operator
WestLaw syntax: `ATLEAST3(procurement)` — operand must appear at least n times on the page.
- **Lexer**: match `ATLEAST\d+` pattern → emit `ATLEAST` token with `count` value
- **Parser**: prefix unary operator at the same precedence level as `NOT`
- **AST**: add `{ type: "atleast"; count: number; operand: QueryNode }` to `QueryNode` union
- **Evaluator**: evaluate operand, check `matchCountOf(inner) >= count`; propagate spans/literals if satisfied
- **AdvancedHelp**: add `{ syntax: 'ATLEAST3(procurement)', desc: 'Term must appear at least 3 times on the page.' }`

**Gate:** `npm test` is green, including all new test cases.

---

### Phase 5 — Frontend

#### `AdminPanel.tsx` (new)
Direct port of the Gradio Admin tab. Three sections:
1. **Index Stats**: button → `GET /api/admin/stats` → renders indexed row count, ready flag, status message, index name
2. **Manual Refresh**: button → `POST /api/admin/refresh` → shows run ID, auto-populates Run ID field
3. **Run Status**: text field (auto-populated from refresh) + button → `GET /api/admin/status/:runId`; polls every 5s while RUNNING/PENDING, auto-stops on TERMINATED

#### `App.tsx` (modified)
- Top-level tabs: **Search** (default, index 0) and **Admin** (index 1)
- Header/StatusBar visible on both tabs
- Search tab: existing three-panel layout
- Admin tab: `<AdminPanel />`

#### `SearchPanel.tsx` (modified)
- **Mode toggle** above the textarea: `Deterministic` (default) | `Semantic`
  - Deterministic: existing behavior, Advanced syntax drawer visible
  - Semantic: hides Advanced syntax drawer (inapplicable to grammar); shows soft AI-similarity hint
- `mode` passed up to `App.tsx` via `onSubmit` callback so the right API route is called
- **Query history**: localStorage key `pa-search-history`, capped at 20 entries; shown as a clickable dropdown when textarea is focused; click to populate and auto-submit

#### `ResultsList.tsx` (modified)
- **Document-level grouping toggle** in results header (default: off)
- When on: group by `documentId`, collapsible document headers with page hits nested inside
- `"semantic"` match type badge with a distinct color (e.g., purple) separate from phrase/proximity/boolean/term

#### `api.ts` (modified)
Add:
```typescript
export async function fetchSemanticSearch(query: string, numResults?: number): Promise<SearchResponse>
export async function fetchAdminStats(): Promise<AdminStats>
export async function triggerRefresh(): Promise<{ runId: string; message: string }>
export async function fetchRunStatus(runId: string): Promise<AdminRunStatus>
```

#### `types.ts` (modified)
```typescript
// Extend existing MatchType
export type MatchType = "phrase" | "proximity" | "boolean" | "term" | "semantic";

// New admin types
export interface AdminStats {
  indexedRowCount: number | string;
  ready: boolean;
  statusMessage: string;
  indexName: string;
}

export interface AdminRunStatus {
  runId: string;
  lifecycleState: string;
  resultState: string;
  statusLabel: string;  // "Running...", "Completed", "Failed", etc.
  startTime: string;
  endTime: string;
  errorMessage?: string;
}
```

**Gate:** Search and Admin tabs fully functional; both search modes return results; query history and grouping toggle work.

---

### Phase 6 — Config & Deploy

#### Updated `app.yaml`
```yaml
command:
  - npm
  - run
  - start
env:
  - name: DATABRICKS_HOST
    value: "https://adb-5672234203219303.3.azuredatabricks.net"
  - name: VS_INDEX_NAME
    value: "tws_ro_region5.rcd.pa_appeals_chunks_vs_index"
  - name: VS_ENDPOINT_NAME
    value: "pa-appeals-search-endpoint"
  - name: PROCESSOR_NOTEBOOK_PATH
    value: "/Users/0492734585@fema.dhs.gov/pa-appeals-and-policy-search/PA Appeals PDF Incremental Processor"
  - name: APPEALS_VOLUME_PATH
    value: "/Volumes/tws_ro_region5/rcd/pa_second_appeals"
  - name: DATABRICKS_WAREHOUSE_ID
    valueFrom:
      resourceRef:
        name: sql-warehouse
  - name: DOCUMENTS_TABLE
    value: ""   # fill in after deterministic index is built
  - name: PAGES_TABLE
    value: ""   # fill in after deterministic index is built
```

> `APPEALS_VOLUME_PATH` is confirmed from the processor notebook. `DOCUMENTS_TABLE` and `PAGES_TABLE` stay blank until `01_build_page_index.py` is run — app runs in Demo mode for deterministic search until then, with semantic search live immediately.

#### Deploy sequence (always in this order)
```bash
# 1. Check status before touching anything
databricks apps get pa-appeals-and-policy-search --output JSON

# 2a. If RUNNING -> deploy directly
databricks apps deploy pa-appeals-and-policy-search \
  --source-code-path /Workspace/Users/0492734585@fema.dhs.gov/pa-appeals-and-policy-search

# 2b. If STOPPED -> start first, then deploy
databricks apps start pa-appeals-and-policy-search --timeout 20m
databricks apps deploy pa-appeals-and-policy-search \
  --source-code-path /Workspace/Users/0492734585@fema.dhs.gov/pa-appeals-and-policy-search
```

## 6. Key Design Decisions

**D1 — User token forwarding (CRITICAL)**
Both the semantic search route and all admin routes must forward `X-Forwarded-Access-Token` from the browser request header. Do **not** substitute the app service principal token. The Python app does this today, and the Node.js app must mirror it exactly. This is required because the Vector Search index enforces per-user data access control, and the Jobs API submissions should run as the calling user, not the app.

**D2 — Notebook path is immutable**
The processor notebook (`PA Appeals PDF Incremental Processor`) stays at its current workspace path. The Power Automate flow (documented in the notebook's final markdown cell) submits it by path — moving or renaming the notebook would silently break all automated SharePoint-triggered PDF ingestion. The path is stored as a single config constant (`PROCESSOR_NOTEBOOK_PATH` in `config.ts`); it must not be hardcoded in multiple places.

**D3 — Search mode toggle placement**
The Deterministic | Semantic toggle lives in `SearchPanel.tsx`, above the query textarea. Deterministic is the default (WestLaw-style, front and center as requested). The Advanced syntax drawer hides entirely in Semantic mode since boolean/proximity grammar does not apply to vector similarity search.

**D4 — Unified result shape**
Semantic results use the same `SearchResult` interface as deterministic results: `matchType: "semantic"`, no `highlightTerms`, raw float score as `score`, `matchCount: 1`. This means `ResultsList.tsx` renders both modes with one component — no conditional branching on result type.

**D5 — Demo mode stays intact**
If `VS_INDEX_NAME` is not set, the semantic search route returns a clearly labeled demo response. Admin routes return stub data. Local development (`npm run dev`) works offline with fabricated data.

**D6 — Hybrid launch state**
The app will go live in a hybrid state: semantic search is immediately functional (VS index is already built and populated with 2,797 documents), while deterministic search runs in Demo mode until `01_build_page_index.py` is run and `DOCUMENTS_TABLE`/`PAGES_TABLE` env vars are set. This is acceptable — users get a working search on day one.

**D7 — WestLaw enhancement scope for Phase 1**
Only wildcard (`term*`) and `ATLEAST(n)` are built now. The following remain deferred:
- **Field search** (`FILE(...)`, `DISASTER(...)`, `STATE(...)`) — blocked until `disaster_number`, `appeal_level`, `state`, `applicant`, `decision_outcome` columns are populated in the Delta index (they exist but are NULL per the README)
- **Date range filtering** — blocked until `decision_date` is reliably sourced from filenames or document metadata
- **Global cross-page match navigation** — listed in the repo README as recommended enhancement #1; deferred to a later phase

## 7. Open Questions

| # | Question | Impact |
| --- | --- | --- |
| OQ1 | What catalog/schema for the deterministic index tables? | Needed to set `DOCUMENTS_TABLE` / `PAGES_TABLE` in `app.yaml` |
| OQ2 | Should the Admin tab be gated to specific users/roles, or visible to all authenticated users? | Access control logic in `AdminPanel.tsx` |
| OQ3 | Is a SQL warehouse already attached to the app as a resource named `sql-warehouse`? | Required for deterministic SQL source; check App resources in the UI |
| OQ4 | Should query history be per-browser only (localStorage) or per-user across devices (server-side)? | `SearchPanel.tsx` implementation scope |
| OQ5 | Is the PowerAutomate flow currently active? | Confirms D2 — the notebook path must never change |

---

## 8. What Is Not Changing

- `PA Appeals PDF Incremental Processor` notebook — no edits, no moves, no renames, ever
- Vector Search index (`tws_ro_region5.rcd.pa_appeals_chunks_vs_index`) and chunks table — read-only, untouched
- App name (`pa-appeals-and-policy-search`) and live URL
- App resources already attached in the Databricks App UI (SQL warehouse, volume)
- PowerAutomate flow if configured (submits notebook by path; path does not change)
- The `Github-pa-appeals-and-policy-search/` folder — remains as a code reference, never deployed

---

## 9. Build Order Summary

```
Phase 1  Scaffold              Author all Node.js/React files directly in workspace folder
                               npm install + npm run build passing in demo mode
Phase 2  Semantic backend      server/src/semantic/vectorSearch.ts
                               POST /api/semantic-search route
Phase 3  Admin backend         GET /api/admin/stats
                               POST /api/admin/refresh
                               GET /api/admin/status/:runId
Phase 4  Query enhancements    Wildcard (*) and ATLEAST(n):
                               queryParser.ts, evaluator.ts, types.ts, tests
Phase 5  Frontend              AdminPanel.tsx (new)
                               App.tsx tabs, SearchPanel mode toggle + query history
                               ResultsList grouping, api.ts, types.ts, styles.css
Phase 6  Config & deploy       Updated app.yaml -> apps get -> apps deploy
-----------------------------------------------------------------------
Deferred  Deterministic index  Run 01_build_page_index.py against the confirmed volume
                               Set DOCUMENTS_TABLE and PAGES_TABLE in app.yaml
Deferred  Field search         After appeal metadata columns are populated in Delta index
Deferred  Date range filter    After decision_date is reliably sourced from documents
```